In [ ]:
import numpy as np
import bacco
import matplotlib.pyplot as plt

import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
def pars(i, mstar):

    arr = np.vstack( ( mstar, np.ones(len(mstar)) * wind_en[i],\
                        np.ones(len(mstar)) * wind_vel[i],\
                        np.ones(len(mstar)) * rho_rec[i],\
                        np.ones(len(mstar)) * sf_ts[i],\
                        np.ones(len(mstar)) * ef_kin[i],\
                        np.ones(len(mstar)) * ef_high[i],\
                        np.ones(len(mstar)) * f_re[i])).T

    return arr

In [ ]:
wind_en_or      = []
wind_vel_or     = []
rho_rec_or      = []
sf_ts_or        = []
ef_kin_or       = []
ef_high_or      = []
f_re_or         = []

for i in range(31):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    else:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"

    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en_or.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel_or.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec_or.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin_or.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re_or.append(float(line.split()[1]))

rho_rec_or = np.log10(rho_rec_or)
ef_kin_or = np.log10(ef_kin_or)
        
wind_en   = (np.asarray(wind_en_or) - np.mean(wind_en_or)) / np.std(wind_en_or)
wind_vel  = (np.asarray(wind_vel_or) - np.mean(wind_vel_or)) / np.std(wind_vel_or)
rho_rec   = (np.asarray(rho_rec_or) - np.mean(rho_rec_or)) / np.std(rho_rec_or)
sf_ts     = (np.asarray(sf_ts_or) - np.mean(sf_ts_or)) / np.std(sf_ts_or)
ef_kin    = (np.asarray(ef_kin_or) - np.mean(ef_kin_or)) / np.std(ef_kin_or)
ef_high   = (np.asarray(ef_high_or) - np.mean(ef_high_or)) / np.std(ef_high_or)
f_re      = (np.asarray(f_re_or) - np.mean(f_re_or)) / np.std(f_re_or)

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME
    
_snap = 264

zoom = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(_snap,_snap), dm_file="snapdir_{:03d}/snapshot_{:03d}".format(_snap,_snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, use_ids=False, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(_snap,_snap), numpart=4320**3)



In [ ]:
# Load halo selection
with open("/cosmos_storage/home/fgmaion/MTNG-resims/halo_selection/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []
    for line in f.readlines():
        final_sel.append(int(line.split()[0]))
final_sel = np.array(final_sel)

In [ ]:
xmatch = utils.cross_match(zoom, snap=264, name='fiducial')

# Load MTNG and get the fraction of halos to do the upweighting
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

zoom_split = utils.split_halos(zoom)

zoom_sel = {}
zoom_sel['sel'] = xmatch['ind'][:,np.newaxis,np.newaxis]
zoom_sel['h_frac'] = h_frac[np.newaxis, :]

In [ ]:
zoom_split = utils.split_halos(zoom)
mtng_split = utils.split_halos(mtng)

In [ ]:
result = zoom_split.smhm_ratio(sel_mask=zoom_sel, nbins=10)

In [ ]:
smhm_mtng = np.array([
[209686269861.9256, 0.015053961766957058],
[225183781637.04245, 0.018450671465512436],
[242723554805.14453, 0.02110298636026753],
[261963946559.15506, 0.02384850461814403],
[282730041087.1567, 0.026609044952692867],
[305142280772.19366, 0.029369585287241734],
[328918005322.34796, 0.032210677612179245],
[351832276645.6902, 0.035102749102005226],
[372852273687.945, 0.03782747155707056],
[394833775486.26953, 0.04043429012915703],
[418421145271.267, 0.04312665734215887],
[446671290702.6251, 0.04589413334025913],
[473357198894.6497, 0.04861577434369935],
[501637920999.32513, 0.051345118976202336],
[533135163206.2874, 0.05418379498857778],
[559475522615.4365, 0.05681053117591567],
[587124295163.5444, 0.05953122362462232],
[621218943625.1672, 0.06235285808200891],
[651915118556.4943, 0.06502657240003121],
[684129982099.9357, 0.0677221370113951],
[724184626258.7635, 0.07070038790400912],
[766580489583.6328, 0.07363857992549686],
[800639835092.4017, 0.07620704211588755],
[842741418544.3407, 0.0788657412830451],
[885567897000.5422, 0.08188366572003555],
[930010973396.6097, 0.08469398214857507],
[971095626218.6653, 0.08761324782676386],
[1025016996367.9647, 0.09072791194423346],
[1073528781623.213, 0.09359453141080254],
[1119036583185.1938, 0.09662514291991581],
[1184183141693.0876, 0.09948829990889121],
[1234563934674.868, 0.10246210879192949],
[1297311098404.0806, 0.10553465896227393],
[1362024190912.7864, 0.10833403513328971],
[1441745142440.645, 0.11118573641030036],
[1526826969118.2478, 0.11422318784699392],
[1616898794324.5774, 0.11711041851696408],
[1712284210304.5132, 0.1199976491869342],
[1813301307494.9028, 0.12290490929246747],
[1920307359324.576, 0.12593234601137943],
[2043280023682.5874, 0.12887419162123717],
[2184324049792.1982, 0.1312905789072713],
[2425152662629.47, 0.13269448292074762],
[2692507157860.8706, 0.13402191090752838],
[2989335437246.93, 0.1353493388943091],
[3348064075775.284, 0.1358440077759344],
[3622297287140.411, 0.13416768530577614],
[3803780525516.044, 0.1320743302783241],
[4096598203050.5093, 0.12977225551538393],
[4344778719253.817, 0.12749746771480136],
[4956733518345.668, 0.12408796387293669],
[5298919457650.736, 0.12122871148684591],
[5851721956313.52, 0.11785097573077627],
[6187915844376.335, 0.11504087213278844],
[6605435030185.395, 0.1127178220114968],
[7000722059883.387, 0.10948414028306852],
[7358101813356.289, 0.10648579433061575],
[7619774737279.447, 0.10405041390171846],
[7953146472090.5, 0.10151116770598251],
[8446378274155.172, 0.09864859794799954],
[8944624961379.455, 0.09577894497770409],
[9465709789812.361, 0.0932081544567239],
[10017087431384.686, 0.09058729034683594],
[10600596347306.32, 0.0879764409547295],
[11218052512814.762, 0.0853355474092784],
[11871504126576.195, 0.08271468329939047],
[12563035327706.941, 0.08010383390728404],
[13304514095469.943, 0.07746606017177698],
[14221595799172.28, 0.07495881212411076],
[15319117910287.434, 0.07238043992155693],
[16522090464509.936, 0.06973303265406328],
[17819478336005.094, 0.06706309227156107],
[19218761250415.17, 0.0644006629273951],
[20728022679892.727, 0.06177578877490994],
[22393808351876.332, 0.05933436032148068],
[24391211514723.18, 0.0568962662584499],
[26757297250629.105, 0.054267698755032706],
[29411356318885.883, 0.05156871153547943],
[32167640574209.258, 0.04911723622037345],
[35850730722752.63, 0.04675312953560001],
[38738059332813.12, 0.04471112868649077],
[42140473212343.445, 0.043258874980578815],
[45766700601386.34, 0.04099079639869868],
[50111501329673.266, 0.03909795981584524],
[55978608813005.945, 0.03704650112263083],
[62124396306362.19, 0.035161935988198445],
[68944630530193.68, 0.033244595413753686],
[76513933466283.58, 0.0313600302793213],
[84915437925089.61, 0.02958471661159684],
[94242343259400.72, 0.028049756170629825],
[104594275488049.19, 0.026558496316345964],
[116082979634476.77, 0.02504538616872054],
[128835224375761.34, 0.023630602341132262],
[142989953415852.1, 0.02230321968691032],
[158700923276517.8, 0.02103046276604234],
[176142542931733.94, 0.019954358485248652],
[195502127615681.1, 0.018921954791138118],
[216988893637111.9, 0.017867700803685965],
[240842202052916.25, 0.01697732401629573],
[267307256828386.22, 0.015781043122123306],
[296684152367232.3, 0.014683088547988055],
[329308798665830.06, 0.014043990134026019],
[365534724926056.7, 0.013699870680175474],
[405727648479723.75, 0.01300614653285953],
[450340045734877.7, 0.012312422385543587],
[499844450049544.5, 0.01140799898100514],
[554770305100380.1, 0.010214839557310118],
[615777769178206.8, 0.009608516583360538],
[673276190393899.5, 0.00997810386874487]])

In [ ]:
zoom.Cosmology.pars['omega_matter'] / zoom.Cosmology.pars['omega_baryon']

In [ ]:
fig, ax = plt.subplots(dpi=100, figsize=(5.5,5))

ax.set_xscale('log')
ax.plot(result['m200c_mean'], result['smhm_mean'] * (zoom.Cosmology.pars['omega_matter'] / zoom.Cosmology.pars['omega_baryon']) )
ax.plot(smhm_mtng[:,0], smhm_mtng[:,1], label='MTNG', color="C3")

ax.set_xlabel(r"$M_{200,c}$ [$M_\odot$]")
ax.set_ylabel("$(M_\\mathrm{star}(r<30\\mathrm{kpc}) / M_\\mathrm{200,c}) \\times (\\Omega_m/\\Omega_b) $")

In [ ]:
rhalf_mtng = np.array([[4057837104866.573, 194.96785114902266],
                        [4905537587625.687, 227.71157232045582],
                        [3519706338365.108, 174.1511986082501],
                        [2843157253562.988, 144.96111476841122],
                        [2351719937567.1143, 122.37967098989976],
                        [1727623610489.3428, 92.28754054723241],
                        [1269353262993.8354, 72.60011169302058],
                        [889478260338.7367, 55.52212603592578],
                        [623320561755.8387, 43.06406559304907],
                        [320989469727.399, 27.41060436590089],
                        [102856541555.42126, 11.425273289813495],
                        [58242789969.03958, 8.02715116055984],
                        [48188487631.18631, 7.271431410919081],
                        [34595468126.34626, 6.493559711833273],
                        [22069041261.777588, 5.880348586352421],
                        [13432394310.281496, 5.794540762590753],
                        [6012787165.015008, 5.7888426301984435],
                        [3029561228.894818, 5.949323482091947],
                        [1263352225.8798437, 6.027299125065391],
                        [502606746.23103434, 6.460012237893936],
                        [165543707.37286395, 7.4275247790808905],
                        [100753318.96335945, 7.216721038307759],
                        [62770650.32984774, 6.535029979995543]])


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

ax.scatter(result['m2half'], 1e3 * np.array(result['rhalf']), color='C0', marker='s', s=1)
ax.plot(result['m2half_mean'], 1e3 * result['rhalf_mean'], color='C0')
ax.plot(rhalf_mtng[:,0], rhalf_mtng[:,1], label='MTNG', color="C3")

ax.set_xlabel(r"$M_{2\mathrm{half}}$ [$M_\odot$]")
ax.set_ylabel(r"$\log_{10} R_{1/2}$ [kpc]")

In [ ]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']# + ['bf_sim'] 

In [ ]:
Nbins_rhalf = 9

# Estimate the size-mass relation
zoom_rhalf = {}

for i in range(len(name_list)):
    zoom_rhalf[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/size_mass_rel/rhalf_m2half_{}_Nbins{:d}.npy".format(name_list[i], Nbins_rhalf), allow_pickle=True)[0]


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

for i in range(25, 26):
    ax.plot(zoom_rhalf[name_list[i]]['m2half_mean'], 1e3 * zoom_rhalf[name_list[i]]['rhalf_mean'], color='gray', marker='o')
    ax.scatter(zoom_rhalf[name_list[i]]['m2half'], np.array( zoom_rhalf[name_list[i]]['rhalf'] ) * 1e3, color='gray', marker='s', s=1)

ax.plot(rhalf_mtng[:,0], rhalf_mtng[:,1], label='MTNG', color="C3")

ax.set_xlabel(r"$M_{2\mathrm{half}}$ [$M_\odot$]")
ax.set_ylabel(r"$\log_{10} R_{1/2}$ [kpc]")

## Load the computed relations

In [ ]:
# Load Size-Mass Relation
Nbins_rhalf = 9

zoom_rhalf = {}
for i in range(len(name_list)):
    zoom_rhalf[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/size_mass_rel/rhalf_m2half_{}_Nbins{:d}.npy".format(name_list[i], Nbins_rhalf), allow_pickle=True)[0]


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

for i in range(len(name_list)-1):
    ax.plot(zoom_rhalf[name_list[i]]['m2half_mean'], 1e3 * zoom_rhalf[name_list[i]]['rhalf_mean'], color='gray',
    alpha=0.5, lw=1, marker='o')

ax.plot(zoom_rhalf['fiducial']['m2half_mean'], 1e3 * zoom_rhalf['fiducial']['rhalf_mean'], label='Fiducial', color='C3', lw=2)
ax.plot(rhalf_mtng[:,0], rhalf_mtng[:,1], label='MTNG', color="C0")

ax.set_xlabel(r"$M_{2\mathrm{half}}$ [$M_\odot$]")
ax.set_ylabel(r"$\log_{10} R_{1/2}$ [kpc]")

ax.legend()


In [ ]:
import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/scripts")
from GP_models import SMF_Model, fgas_Model
import torch
import gpytorch

In [ ]:
model_rhalf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_model_size_mass.pth")
likelihood_rhalf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_likelihood_size_mass.pth")

model_rhalf.eval()
likelihood_rhalf.eval()

In [ ]:
# Define the training set
train_sel = np.arange(31)

In [ ]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(2, 3, figsize=(15, 10), dpi=100)
    plt.subplots_adjust(wspace=0.15, hspace=0.3)

    for i in range(2):
        for j in range(3):
            mask = ~np.isnan(zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean']) & ~np.isnan(zoom_rhalf[name_list[train_sel[3*i+j]]]['rhalf_mean'])
            test_x = torch.asarray(pars(train_sel[3*i+j], np.log10(zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean'][mask])), dtype=torch.float)
            observed_pred = likelihood_rhalf(model_rhalf(torch.asarray(test_x, dtype=torch.float)))
        
            ax[i,j].plot(np.log10(rhalf_mtng[:,0]), np.log10(rhalf_mtng[:,1] / 1e3), color="C3", lw=3)

            ax[i,j].set_xlim(8.5,13)
            ax[i,j].set_ylim(-2.5, -1)

            ax[i,j].set_title('Simulation {:s}'.format(name_list[train_sel[3*i+j]]), fontsize=16)

            if j==0:
                ax[i,j].set_ylabel('$\log_{10}(R_{1/2}/\mathrm{Mpc})$', fontsize=16)
            
            ax[i,j].set_xlabel('$\log_{10}(M_{2\mathrm{half}}/M_\odot)$', fontsize=16)

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()

            # Shade between the lower and upper confidence bounds
            ax[i,j].fill_between(test_x[:,0].numpy(), observed_pred.mean.numpy()-observed_pred.stddev.numpy(), observed_pred.mean.numpy()+observed_pred.stddev.numpy(), color='C0', alpha=0.4, edgecolor=None)
            ax[i,j].plot(test_x[:,0], observed_pred.mean.numpy(), 'C0', lw=2, label="GP Prediction", alpha=0.9)

            # Plot test data as blue squares
            ax[i,j].plot(np.log10(zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean'][mask]), np.log10(zoom_rhalf[name_list[train_sel[3*i+j]]]['rhalf_mean'][mask]), 's', color="C0", label="Test Data")

ax[0,0].legend(loc='lower left', fontsize=12)

plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/results/testing_plots/size_mass_test.pdf", bbox_inches='tight')

In [ ]:
zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean']